# Hands-on 27 (Bab 11): Ekspor Model Sederhana ke Format Ringan

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Dua jalur deployment edge yang realistis:
- **Jalur A:** Random Forest scikit-learn -> joblib (untuk Raspberry Pi dengan Python).
- **Jalur B:** MLP kecil Keras -> .keras -> TensorFlow Lite (untuk inferensi edge).

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

## 1. Jalur A: Ekspor scikit-learn (joblib)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/export_sklearn_model.py"], capture_output=True, text=True)
print(r.stdout)

## 2. Jalur B: Keras -> TFLite

In [ ]:
r = subprocess.run([sys.executable, "scripts/export_keras_to_tflite.py"], capture_output=True, text=True)
print(r.stdout)

## 3. Verifikasi Inferensi dari File Model

In [ ]:
import joblib
import numpy as np
import pandas as pd

# Inferensi ulang dari model joblib
model = joblib.load("outputs/models/room_status_rf.joblib")
X_test = pd.read_csv("data/processed/classification_status/X_test.csv")
print("[INFO] Prediksi joblib (5 baris):", list(model.predict(X_test.head())))

# Inferensi ulang dari model TFLite
import tensorflow as tf
interpreter = tf.lite.Interpreter(model_path="outputs/models/room_binary.tflite")
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]["index"]
out = interpreter.get_output_details()[0]["index"]
Xb = pd.read_csv("data/processed/classification_room/X_test.csv").values.astype("float32")[:5]
interpreter.set_tensor(inp, Xb)
interpreter.invoke()
print("[INFO] Probabilitas TFLite (5 baris):", interpreter.get_tensor(out).flatten().round(3).tolist())

## Penutup

Training dan deployment adalah dua tahap berbeda. Model joblib (sekitar 425 KB) siap
untuk Raspberry Pi dengan Python biasa; model TFLite hanya sekitar 3.9 KB dan bisa
berjalan dengan interpreter ringan tanpa TensorFlow penuh. Keduanya terverifikasi bisa
dimuat ulang dan dipakai inferensi dari file hasil ekspor di `outputs/models/`.